# AI Power Portfolio — Colab

Pon la URL de tu repo abajo y corre todas las celdas en orden (Runtime > Run all).

In [ ]:
REPO_URL = "https://github.com/r0988137/Capstone.git"


In [ ]:
import subprocess, os

repo_name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
subprocess.run(f"rm -rf {repo_name}", shell=True)
r = subprocess.run(f"timeout 120 git clone --depth 1 --progress {REPO_URL}", shell=True)
assert r.returncode == 0, "El clone falló o tardó más de 2 minutos. Revisa que el repo sea público y que no tenga archivos pesados."

# encuentra la carpeta que tenga src/ y requirements.txt, sin importar el nivel
project_root = None
for dirpath, dirnames, filenames in os.walk(repo_name):
    if "src" in dirnames and "requirements.txt" in filenames and os.path.isdir(os.path.join(dirpath, "src", "workflow")):
        project_root = dirpath
        break

assert project_root is not None, "No se encontró src/workflow en el repo. Faltan archivos nuevos en GitHub (¿subiste el .zip sin descomprimir?)"
os.chdir(project_root)
print("Proyecto en:", os.getcwd())

!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/ai_power_portfolio_data"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["WAREHOUSE_DB_PATH"] = f"{DRIVE_DIR}/warehouse.db"


In [ ]:
!python -m src.db.init_db
!python -m src.etl.load_market_data


In [ ]:
import sqlite3, pandas as pd

conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
df = pd.read_sql("""
    SELECT a.ticker, d.full_date, f.close
    FROM fact_market_price f
    JOIN dim_asset a ON a.asset_key = f.asset_key
    JOIN dim_date d ON d.date_key = f.date_key
    ORDER BY d.full_date DESC
    LIMIT 20
""", conn)
conn.close()
df


In [ ]:
!python -m src.etl.validate_market_data


## Cuestionario + flujo completo (Semanas 5–10)

Opcional: guarda tu `ANTHROPIC_API_KEY` en Colab (icono de llave > Secrets). Si no la pones, todo funciona con la plantilla de texto. El cuestionario se contesta una sola vez, en la última celda de abajo.

In [ ]:
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key cargada: se usará el LLM")
except Exception:
    print("Sin API key: se usará la plantilla (todo funciona igual)")

In [ ]:
from src.workflow.run_workflow import run_full_workflow, format_report

answers = collect_answers_interactive()
conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
result = run_full_workflow(conn, "demo-user", answers)   # include_news=False si quieres saltar noticias
conn.close()
print(format_report(result))

In [ ]:
conn = sqlite3.connect(os.environ["WAREHOUSE_DB_PATH"])
key = result["user_profile_key"]
pd.read_sql("""
    SELECT s.scenario_name, f.horizon_year, f.projected_value
    FROM fact_forecast f JOIN dim_scenario s ON s.scenario_key = f.scenario_key
    WHERE f.user_profile_key = ? ORDER BY f.horizon_year, s.scenario_name
""", conn, params=(key,)).pivot(index="horizon_year", columns="scenario_name", values="projected_value").plot(title="Proyección")

## Semanas 11–12: app web + dashboard

Esta celda abre la app (formulario, resultados y dashboard). Busca el link que dice **Running on public URL** y ábrelo. Si ya tienes datos cargados (ETL), el dashboard los muestra.

In [ ]:
from src.app.app import launch
launch(share=True)   # abre la app; busca el link "Running on public URL"